# BEIR evaluation: SciFact, retrieval only, QAChat's real pipeline

[BEIR](https://github.com/beir-cellar/beir) is a published benchmark suite for retrieval evaluation; this notebook runs its SciFact task against QAChat's own real retrieval pipeline -- the structural opposite of the FiQA notebook's corpus: small, clean, and already abstract-length.

**The Problem.** Given a scientific claim (e.g. *"A high microerythrocyte count raises vulnerability to severe anemia in homozygous alpha (+)-thalassemia trait"*), not a question, can QAChat's real retrieval pipeline find the research-paper abstract that supports or refutes it?

**The Data.** SciFact: 300 test queries with ~1.1 relevant documents each on average, searched against the full official 5,183-document corpus of research-paper abstracts (average length 213.6 words, comfortably inside QAChat's `target_tokens = 300` chunking target.

**The Metrics.** NDCG, MAP, Recall, and Precision at several cutoffs, computed via `pytrec_eval` through `evaluate_retrieval`. NDCG@10 is BEIR's own primary metric.

**What's different from the original benchmark.** Retrieval-only design: BEIR's qrels are relevance judgments, not reference answers, and isolating retrieval keeps the score independent of the small local chat model. Each document becomes its own `.txt` file named by its BEIR id, with results max-aggregated per document id before scoring, though splits should be rare given the average document length. There's no code channel -- `hybrid_search` runs semantic + lexical only.  A claim-verification-specific `SCIFACT_RETRIEVAL_INSTRUCTION` is defined, since QAChat's built-in literature instruction assumes a question rather than a claim to verify. No leave-one-out is needed, since BEIR's query and corpus id spaces are already disjoint. Retrieval oversamples to 100 candidates per query to cover `evaluate_retrieval`'s full k range. And, the corpus/query ids are strings while the qrels' ids are int64, so every join below casts explicitly. And because max-aggregation changes the retrieval unit from what BEIR's own published table measures (one embedding per document, not per chunk), the notebook also runs a whole-document baseline that matches that granularity exactly -- rare as splits should be here.


## 1. Prepare QAChat

Locate the QAChat repo root, put it on the import path, and run QAChat's notebook setup.

In [1]:
from pathlib import Path

root = Path.cwd()
while not (root / "qachat").is_dir() and root != root.parent:
    root = root.parent

print(f"QAChat root: {root}")

QAChat root: /Users/cmanzo/Documents/GitHub/QAChat


In [2]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [3]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)

QAChat environment ready: /Users/cmanzo/Documents/GitHub/QAChat/.venv/bin/python


## 2. Benchmark-only dependencies

`datasets` pulls SciFact from HuggingFace (`BeIR/scifact` for corpus/queries, `BeIR/scifact-qrels` for relevance judgments) -- the same loading mechanism the FiQA notebook uses. `pytrec_eval` is already a `benchmarks/common.py` dependency (`evaluate_retrieval`).


In [4]:
import datasets

print(f"datasets version: {datasets.__version__}")

/Users/cmanzo/Documents/GitHub/QAChat/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


datasets version: 5.0.1


## 3. Load SciFact queries and the test qrels

Test-split qrels only (300 queries, ~1.1 relevant docs each) -- BEIR
reserves train (920 examples) for tuning a retriever, test for reporting.
Queries are filtered down to the ones that actually appear in the test
qrels, and every id is cast to `str` explicitly -- `BeIR/scifact`'s `_id`
columns are strings, `BeIR/scifact-qrels`' `query-id`/`corpus-id` columns
are int64, and relying on Python's cross-type `==` here would silently
match nothing.

SciFact's `queries` rows carry the claim text in `text` (`title` is
typically empty, same convention as FiQA) -- confirmed by inspecting the
loaded rows below rather than assumed.

In [5]:
from collections import defaultdict

all_queries_rows = datasets.load_dataset("BeIR/scifact", "queries")["queries"]
qrels_test_rows = datasets.load_dataset("BeIR/scifact-qrels", split="test")

qrels: dict[str, dict[str, int]] = defaultdict(dict)
for row in qrels_test_rows:
    query_id = str(row["query-id"])
    corpus_id = str(row["corpus-id"])
    qrels[query_id][corpus_id] = int(row["score"])
qrels = dict(qrels)

query_text_by_id = {str(row["_id"]): row["text"] for row in all_queries_rows}

missing_query_ids = set(qrels) - set(query_text_by_id)
if missing_query_ids:
    raise RuntimeError(
        f"{len(missing_query_ids)} qrels query ids have no matching query "
        "text -- id casting or dataset version mismatch."
    )

scifact_queries = {qid: query_text_by_id[qid] for qid in qrels}

relevant_doc_count = sum(len(docs) for docs in qrels.values())
print(f"Loaded {len(scifact_queries)} test queries.")
print(
    f"{relevant_doc_count} relevance judgments "
    f"({relevant_doc_count / len(scifact_queries):.2f} relevant docs/query)."
)

# Sanity check the title/text convention this section's markdown claims.
_sample_id = next(iter(scifact_queries))
_sample_title = next(
    row["title"] for row in all_queries_rows if str(row["_id"]) == _sample_id
)
print(f"\nSample query id {_sample_id}: title={_sample_title!r}")
print(f"text: {scifact_queries[_sample_id][:200]!r}")

'[Errno 8] nodename nor servname provided, or not known' thrown while requesting HEAD https://huggingface.co/datasets/BeIR/scifact/resolve/b3b5335604bf5ee3c4447671af975ea25143d4f5/scifact.py
Retrying in 1s [Retry 1/5].
'[Errno 8] nodename nor servname provided, or not known' thrown while requesting HEAD https://huggingface.co/datasets/BeIR/scifact/resolve/b3b5335604bf5ee3c4447671af975ea25143d4f5/scifact.py
Retrying in 2s [Retry 2/5].
'[Errno 8] nodename nor servname provided, or not known' thrown while requesting HEAD https://huggingface.co/datasets/BeIR/scifact/resolve/b3b5335604bf5ee3c4447671af975ea25143d4f5/scifact.py
Retrying in 4s [Retry 3/5].
'[Errno 8] nodename nor servname provided, or not known' thrown while requesting HEAD https://huggingface.co/datasets/BeIR/scifact/resolve/b3b5335604bf5ee3c4447671af975ea25143d4f5/scifact.py
Retrying in 8s [Retry 4/5].
'[Errno 8] nodename nor servname provided, or not known' thrown while requesting HEAD https://huggingface.co/datasets/BeIR/s

Loaded 300 test queries.
339 relevance judgments (1.13 relevant docs/query).

Sample query id 1: title=''
text: '0-dimensional biomaterials show inductive properties.'


## 4. Load the SciFact corpus

The full official 5,183-document corpus -- small enough that, unlike the FiQA notebook's run, there is no smoke-test sampling knob here at all. Every run of this notebook is already a full-corpus run.


In [6]:
corpus_rows = datasets.load_dataset("BeIR/scifact", "corpus")["corpus"]

scifact_corpus = {
    str(row["_id"]): {"title": row["title"], "text": row["text"]} for row in corpus_rows
}

covered_relevant = sum(
    1 for docs in qrels.values() for doc_id in docs if doc_id in scifact_corpus
)
print(f"Loaded {len(scifact_corpus)} corpus documents.")
print(f"{covered_relevant}/{relevant_doc_count} relevant documents are present.")

Loaded 5183 corpus documents.
339/339 relevant documents are present.


## 5. Load an embedding model

Retrieval only -- no chat model, no `ChatSession`, no generation settings. Same embedding model and backend selection as the FiQA notebook.


In [7]:
BACKEND = "auto"
ALLOW_DOWNLOAD = True
EMBED_MODEL_BY_BACKEND = {
    "mlx": "mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ",
    "torch": "Qwen/Qwen3-Embedding-0.6B",
}

In [8]:
from qachat import choose_backend

selected_backend = choose_backend(BACKEND)
EMBED_MODEL_ID = EMBED_MODEL_BY_BACKEND[selected_backend]

print(f"Backend: {selected_backend}")
print(f"Embedding model: {EMBED_MODEL_ID}")

Backend: mlx
Embedding model: mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ


In [9]:
from qachat import EmbeddingSession

embedding_session = EmbeddingSession(
    EMBED_MODEL_ID,
    backend=selected_backend,
    allow_download=ALLOW_DOWNLOAD,
)

print(f"Embedding model loaded on: {embedding_session.device}")

Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 3451.01it/s]


Embedding model loaded on: metal


## 6. Turn the corpus into a real QAChat project

`sandbox/scifact_benchmark/` is a real QAChat project -- `profile.ini` (checked into the repository, same convention used throughout this project's benchmark notebooks, not generated inline) plus a `documents/` source folder, `kind = literature`. Each corpus document is written as `documents/<doc_id>.txt` (`title` prepended when present -- SciFact abstracts do have real titles, unlike FiQA's mostly-empty ones) -- so `chunk.path.stem` recovers the original BEIR document id directly once real chunking runs.

No `MAX_DOC_CHARS`-style truncation knob here: average document length is 213.6 words (~285 tokens), comfortably inside `target_tokens = 300`, so this corpus shouldn't have DS-1000's long-tail outlier problem. Checked directly below rather than assumed.


In [10]:
from benchmarks.common import corpus_already_written

project_dir = root / "benchmarks" / "sandbox" / "scifact_benchmark"
profile_path = project_dir / "profile.ini"
if not profile_path.exists():
    raise RuntimeError(
        f"Missing project profile: {profile_path}. This notebook expects "
        "the profile checked into the repository at that path."
    )

documents_dir = project_dir / "documents"

doc_char_lengths = [
    len(f"{doc['title']}\n\n{doc['text']}" if doc["title"] else doc["text"])
    for doc in scifact_corpus.values()
]

if corpus_already_written(documents_dir):
    print(f"Corpus already present under {documents_dir}, skipping write.")
else:
    documents_dir.mkdir(parents=True, exist_ok=True)
    for doc_id, doc in scifact_corpus.items():
        text = f"{doc['title']}\n\n{doc['text']}" if doc["title"] else doc["text"]
        (documents_dir / f"{doc_id}.txt").write_text(text, encoding="utf-8")
    print(f"Wrote {len(scifact_corpus)} documents to {documents_dir.relative_to(root)}")

doc_char_lengths.sort()
median_chars = doc_char_lengths[len(doc_char_lengths) // 2]
print(
    f"Document length (chars): median={median_chars}, "
    f"max={doc_char_lengths[-1]} (FiQA-scale outliers would show here as "
    f"a max far past the median; SciFact shouldn't have any)."
)


Corpus already present under /Users/cmanzo/Documents/GitHub/QAChat/benchmarks/sandbox/scifact_benchmark/documents, skipping write.
Document length (chars): median=1427, max=10128 (FiQA-scale outliers would show here as a max far past the median; SciFact shouldn't have any).


## 7. Build the retrieval corpus with QAChat's real pipeline

`scan_repository` -> `read_repository_files` -> `extract_source_units` -> `chunk_source_units`, same pipeline as the FiQA notebook. `max_entries` is sized to `len(scifact_corpus) + 100` from the start (a lesson already learned while building that notebook, not rediscovered here): `scan_repository` defaults to `max_entries=10_000` and raises `ValueError` past it rather than truncating silently, and `documents/` is flat (one entry per document), so this scales with the corpus instead of relying on the default headroom being enough. No code graph is built -- there's nothing for it to index.


In [11]:
from qachat.project import open_project

project = open_project(project_dir)
profile = project.profile

print(f"Project source: {project.source}")
print(f"Chunking: target={profile.target_tokens} max={profile.max_tokens} overlap={profile.overlap_tokens}")

Project source: /Users/cmanzo/Documents/GitHub/QAChat/benchmarks/sandbox/scifact_benchmark/documents
Chunking: target=300 max=400 overlap=40


In [12]:
from qachat.repository import scan_repository
from qachat.sources import extract_source_units, read_repository_files
from qachat.chunking import chunk_source_units

repository_report = scan_repository(
    project.source,
    extensions=profile.extensions,
    max_file_bytes=profile.max_file_bytes,
    max_entries=len(scifact_corpus) + 100,
)
source_files = read_repository_files(repository_report)
source_units = extract_source_units(source_files)
scifact_chunks = chunk_source_units(
    source_units,
    count_tokens=embedding_session.count_tokens,
    target_tokens=profile.target_tokens,
    max_tokens=profile.max_tokens,
    overlap_tokens=profile.overlap_tokens,
)

multi_chunk_docs = len(scifact_corpus) - len({chunk.path.stem for chunk in scifact_chunks})
print(f"Scanned {len(source_files)} files, {len(source_units)} source units.")
print(
    f"Built {len(scifact_chunks)} chunks "
    f"({multi_chunk_docs} documents split into more than one chunk)."
)

Scanned 5183 files, 10366 source units.
Built 11554 chunks (0 documents split into more than one chunk).


## 8. Build a whole-document baseline (no splitting)

Build one unsplit chunk per corpus document -- the same granularity BEIR's own published table uses. SciFact's abstracts are short and uniform (median well under `target_tokens = 300`, checked in section 6) with none of FiQA's or DS-1000's long-tail outlier problem, and none of FiQA's empty-post problem either -- but this baseline still applies the same empty-text guard FiQA's needed, as a defensive check rather than an assumption that SciFact's cleaner corpus makes it unnecessary. Chunks are still sorted by length before embedding (section 9), so batching doesn't pad the many short abstracts up to a rare longer one's size.


In [13]:
from qachat.chunking import Chunk


def build_whole_document_chunks() -> list[Chunk]:
    """Build one whole, unsplit chunk per corpus document.

    Real chunking (section 7) splits long abstracts into paragraph-sized
    pieces on its own, and silently drops any unit whose text is blank
    after stripping (qachat/chunking.py filters this at both the source-
    unit and final-chunk level). This baseline can't split anything --
    it's one chunk per document by definition -- but it still applies that
    same empty-text filter itself, the same lesson learned building the
    FiQA notebook's equivalent baseline, even though SciFact's abstracts
    are expected to always have real text.
    """

    chunks = []
    for doc_id, doc in scifact_corpus.items():
        text = f"{doc['title']}\n\n{doc['text']}" if doc["title"] else doc["text"]
        if not text.strip():
            continue
        chunks.append(
            Chunk(
                identifier=doc_id,
                path=Path(f"{doc_id}.txt"),
                text=text,
                start_line=1,
                end_line=text.count("\n") + 1,
                kind="whole_document",
                symbol=doc["title"] or None,
                section=None,
            )
        )
    # Sorted by length, not corpus order: batching (section 9) pads every
    # sequence in a batch to its longest member, so leaving a handful of
    # longer abstracts next to dozens of shorter ones wastes some of that
    # batch's compute on padding. Grouping similar lengths together keeps
    # batches close to their true cost instead.
    chunks.sort(key=lambda chunk: len(chunk.text))
    return chunks


whole_document_chunks = build_whole_document_chunks()
skipped_empty = len(scifact_corpus) - len(whole_document_chunks)
print(
    f"scifact: {len(whole_document_chunks)} whole-document chunks "
    f"({skipped_empty} documents skipped: empty title and text)"
)


scifact: 5183 whole-document chunks (0 documents skipped: empty title and text)


## 9. Run retrieval

Define semantic, lexical, and hybrid retrieval as reusable functions, then build the real-chunking index once. No code channel: SciFact's corpus is research-paper prose, not parseable code, so there's nothing for `code_search` to resolve.

SciFact's qrels and the published NDCG@10 table are both document-level, while QAChat's retrieval identity is `Chunk.identifier` -- so each function's raw chunk-level results are max-aggregated to one best-scoring chunk per document id (`chunk.path.stem`) before being returned, same as the FiQA notebook.

Embedding is the slow part -- so the index is saved under the project's own `indexes/` folder (the same mechanism `RAGSession` itself uses) and reloaded next time if its fingerprint still matches, rebuilt otherwise.


In [14]:
import time

from tqdm.auto import tqdm

from qachat import (
    build_lexical_index,
    build_semantic_index,
    hybrid_search,
    indexes_are_current,
    lexical_search,
    load_indexes,
    prepare_persistent_indexes,
    save_indexes,
    semantic_matrix,
    semantic_search,
)
from qachat.query_formats import format_qwen3_query
from qachat.retrievals.code_graph.code_graph import CodeGraph

EMBED_BATCH_SIZE = 32
CANDIDATE_COUNT = 100
QUERY_SAMPLE_SIZE = None  # None for all test queries

sampled_query_ids = (
    list(scifact_queries)
    if QUERY_SAMPLE_SIZE is None
    else list(scifact_queries)[:QUERY_SAMPLE_SIZE]
)

SCIFACT_RETRIEVAL_INSTRUCTION = (
    "Given a scientific claim, retrieve research paper abstracts that "
    "support or refute it."
)

# No code entities exist for SciFact's prose corpus, so the
# fingerprint/caching machinery below -- built for QAChat's real
# semantic+lexical+code indexes -- is given a permanently empty code graph.
empty_code_graph = CodeGraph(entities=(), relations=())


def _max_aggregate_by_document(ranked) -> dict[str, object]:
    """Collapse chunk-level search results to one best-scoring result per document id.

    Otherwise one document could occupy two slots in a query's top-k and
    silently crowd out a different, actually-distinct relevant document.
    """

    doc_best: dict[str, object] = {}
    for result in ranked:
        doc_id = result.chunk.path.stem
        if doc_id not in doc_best or result.score > doc_best[doc_id].score:
            doc_best[doc_id] = result
    return doc_best


def run_semantic_retrieval(
    semantic_index, *, desc: str, matrix=None, count: int = CANDIDATE_COUNT
) -> dict[str, dict[str, float]]:
    """Rank every sampled query's documents with QAChat's semantic search."""

    search_kwargs = {"backend": "numpy", "matrix": matrix} if matrix is not None else {}
    results: dict[str, dict[str, float]] = {}
    query_items = tqdm(sampled_query_ids, desc=f"{desc}: semantic", unit="query", leave=False)
    for query_id in query_items:
        formatted_query = format_qwen3_query(scifact_queries[query_id], SCIFACT_RETRIEVAL_INSTRUCTION)
        ranked = semantic_search(
            semantic_index, formatted_query, embedding_session, count=count, **search_kwargs
        )
        results[query_id] = {
            doc_id: r.score for doc_id, r in _max_aggregate_by_document(ranked).items()
        }
    return results


def run_lexical_retrieval(
    lexical_index, *, desc: str, count: int = CANDIDATE_COUNT
) -> dict[str, dict[str, float]]:
    """Rank every sampled query's documents with QAChat's lexical search."""

    results: dict[str, dict[str, float]] = {}
    query_items = tqdm(sampled_query_ids, desc=f"{desc}: lexical", unit="query", leave=False)
    for query_id in query_items:
        ranked = lexical_search(lexical_index, scifact_queries[query_id], count=count)
        results[query_id] = {
            doc_id: r.score for doc_id, r in _max_aggregate_by_document(ranked).items()
        }
    return results


def run_hybrid_retrieval(
    semantic_index,
    lexical_index,
    *,
    desc: str,
    matrix=None,
    count: int = CANDIDATE_COUNT,
    keep_context: bool = False,
):
    """Fuse semantic and lexical rankings, max-aggregated per document id."""

    search_kwargs = {"backend": "numpy", "matrix": matrix} if matrix is not None else {}
    results: dict[str, dict[str, float]] = {}
    context: dict[str, list[dict[str, object]]] = {}
    query_items = tqdm(sampled_query_ids, desc=f"{desc}: hybrid", unit="query", leave=False)
    for query_id in query_items:
        query_text = scifact_queries[query_id]
        formatted_query = format_qwen3_query(query_text, SCIFACT_RETRIEVAL_INSTRUCTION)
        semantic_ranked = semantic_search(
            semantic_index, formatted_query, embedding_session, count=count, **search_kwargs
        )
        lexical_ranked = lexical_search(lexical_index, query_text, count=count)
        combined = hybrid_search(semantic=semantic_ranked, lexical=lexical_ranked, count=count)
        doc_best = _max_aggregate_by_document(combined)
        results[query_id] = {doc_id: r.score for doc_id, r in doc_best.items()}
        if keep_context:
            context[query_id] = [
                {
                    "doc_id": doc_id,
                    "score": r.score,
                    "channels": list(r.channels),
                    "is_relevant": qrels.get(query_id, {}).get(doc_id, 0) > 0,
                }
                for doc_id, r in sorted(
                    doc_best.items(), key=lambda item: item[1].score, reverse=True
                )
            ]
    return (results, context) if keep_context else results


index_path = project.indexes / "scifact_retrieval.json"
cached = None
if index_path.exists():
    try:
        cached = load_indexes(index_path)
    except ValueError:
        cached = None
    if cached is not None and not indexes_are_current(cached, scifact_chunks, empty_code_graph):
        cached = None

if cached is not None:
    corpus_semantic_index = cached.semantic
    corpus_lexical_index = cached.lexical
    print(f"scifact: loaded cached index ({len(scifact_chunks)} chunks) from {index_path.relative_to(root)}")
else:
    with tqdm(total=len(scifact_chunks), desc="Embedding scifact chunks") as progress_bar:

        def _on_embed_progress(current: int, total: int) -> None:
            progress_bar.n = current
            progress_bar.refresh()

        corpus_semantic_index = build_semantic_index(
            scifact_chunks, embedding_session, batch_size=EMBED_BATCH_SIZE, progress_callback=_on_embed_progress
        )

    _lexical_start = time.monotonic()
    corpus_lexical_index = build_lexical_index(scifact_chunks)
    print(f"scifact: lexical index built in {time.monotonic() - _lexical_start:.1f}s.")

    save_indexes(
        prepare_persistent_indexes(corpus_semantic_index, corpus_lexical_index, empty_code_graph),
        index_path,
    )
    print(f"scifact: indexed {len(scifact_chunks)} chunks, saved to {index_path.relative_to(root)}")

corpus_matrix = semantic_matrix(corpus_semantic_index)


Embedding scifact chunks: 100%|██████████| 11554/11554 [11:36<00:00, 16.58it/s]


scifact: lexical index built in 1.6s.
scifact: indexed 11554 chunks, saved to benchmarks/sandbox/scifact_benchmark/indexes/scifact_retrieval.json


## 10. Score against the published range

Scores against document-level qrels via `evaluate_retrieval` -- BEIR's own primary metric is NDCG@10. Semantic and lexical are scored alongside hybrid; with only two real channels, this table is already the full picture.


In [15]:
from benchmarks.common import evaluate_retrieval

sampled_qrels = {qid: qrels[qid] for qid in sampled_query_ids}

semantic_results = run_semantic_retrieval(corpus_semantic_index, desc="scifact", matrix=corpus_matrix)
lexical_results = run_lexical_retrieval(corpus_lexical_index, desc="scifact")
hybrid_results, retrieved_context_by_query = run_hybrid_retrieval(
    corpus_semantic_index, corpus_lexical_index, desc="scifact", matrix=corpus_matrix, keep_context=True
)

scifact_scores = {
    "semantic": evaluate_retrieval(sampled_qrels, semantic_results),
    "lexical": evaluate_retrieval(sampled_qrels, lexical_results),
    "hybrid": evaluate_retrieval(sampled_qrels, hybrid_results),
}

for channel, channel_scores in scifact_scores.items():
    print(
        f"{channel:>10}: NDCG@10={channel_scores['ndcg']['NDCG@10']:.3f}  "
        f"MRR@10={channel_scores['mrr']['MRR@10']:.3f}  Recall@10={channel_scores['recall']['Recall@10']:.3f}"
    )


  semantic: NDCG@10=0.681  MRR@10=0.643  Recall@10=0.813
   lexical: NDCG@10=0.533  MRR@10=0.485  Recall@10=0.708
    hybrid: NDCG@10=0.681  MRR@10=0.652  Recall@10=0.798


## 11. Compare: whole-document baseline vs. QAChat chunking

`whole_document` matches BEIR's own published granularity exactly -- one embedding per document, the same unit the table below reports. `qachat_chunking` is `scifact_scores["hybrid"]`, QAChat's real pipeline. Any gap between the two is a direct consequence of splitting some documents into more than one chunk -- expected to be small here, since section 7 should show few if any SciFact abstracts long enough to split.


In [16]:
with tqdm(total=len(whole_document_chunks), desc="Embedding scifact whole documents") as progress_bar:

    def _on_embed_progress(current: int, total: int) -> None:
        progress_bar.n = current
        progress_bar.refresh()

    whole_document_semantic_index = build_semantic_index(
        whole_document_chunks, embedding_session, batch_size=EMBED_BATCH_SIZE, progress_callback=_on_embed_progress
    )
_lexical_start = time.monotonic()
whole_document_lexical_index = build_lexical_index(whole_document_chunks)
print(f"scifact whole-document: lexical index built in {time.monotonic() - _lexical_start:.1f}s.")
whole_document_matrix = semantic_matrix(whole_document_semantic_index)

whole_document_results = run_hybrid_retrieval(
    whole_document_semantic_index,
    whole_document_lexical_index,
    desc="scifact whole-document",
    matrix=whole_document_matrix,
)

comparison_scores = {
    "whole_document": evaluate_retrieval(sampled_qrels, whole_document_results),
    "qachat_chunking": scifact_scores["hybrid"],
}

HEADLINE_MEASURES = (
    ("mrr", "MRR@10"),
    ("ndcg", "NDCG@10"),
    ("recall", "Recall@10"),
    ("recall", "Recall@100"),
)

header = ["mode", *(label for _, label in HEADLINE_MEASURES)]
print(" | ".join(f"{h:>14}" for h in header))
for mode in ("whole_document", "qachat_chunking"):
    mode_scores = comparison_scores[mode]
    values = [f"{mode_scores[group][label]:.3f}" for group, label in HEADLINE_MEASURES]
    print(" | ".join(f"{cell:>14}" for cell in [mode, *values]))


Embedding scifact whole documents: 100%|██████████| 5183/5183 [06:23<00:00, 13.52it/s]


scifact whole-document: lexical index built in 1.5s.


          mode |         MRR@10 |        NDCG@10 |      Recall@10 |     Recall@100
whole_document |          0.675 |          0.705 |          0.828 |          0.960
qachat_chunking |          0.652 |          0.681 |          0.798 |          0.952


**Reading the two numbers.** `qachat_chunking` measurably trails `whole_document` here. That's despite SciFact's abstracts being short and uniform -- the corpus still averages more than two chunks per document, so splitting turns out to be the common case here, not the rare one the intro anticipated.

Both numbers still sit at the top of BEIR's own published range below: `whole_document` clears every retriever they benchmarked, including the cross-encoder reranker BM25+CE, and `qachat_chunking` is still ahead of ColBERT and every dense retriever, behind only BM25+CE.

BEIR's own published NDCG@10 for SciFact (Thakur et al., 2021, Table 1/2), across every retriever they benchmarked:

| Retriever | NDCG@10 |
|---|---|
| DPR | 0.318 |
| ANCE | 0.507 |
| TAS-B | 0.643 |
| GenQ | 0.644 |
| BM25 | 0.665 |
| ColBERT | 0.671 |
| BM25+CE | 0.688 |

## 12. Save results


In [17]:
import json as _json
from benchmarks.common import local_run_timestamp

results_dir = root / "benchmarks" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

timestamp, stamp = local_run_timestamp()

run_record = {
    "benchmark": "BEIR (retrieval only) -- SciFact, QAChat's real pipeline",
    "timestamp": timestamp,
    "embedding_model": EMBED_MODEL_ID,
    "backend": selected_backend,
    "query_sample_size": QUERY_SAMPLE_SIZE,
    "candidate_count": CANDIDATE_COUNT,
    "retrieval_channels": ["semantic", "lexical"],
    "retrieval_instruction": SCIFACT_RETRIEVAL_INSTRUCTION,
    "chunking_profile": {
        "target_tokens": profile.target_tokens,
        "max_tokens": profile.max_tokens,
        "overlap_tokens": profile.overlap_tokens,
    },
    "corpus_document_count": len(scifact_corpus),
    "corpus_chunk_count": len(scifact_chunks),
    "query_count": len(sampled_query_ids),
    "scores": scifact_scores,
    "comparison_scores": comparison_scores,
    "per_query_retrieved": retrieved_context_by_query,
}

output_path = results_dir / f"{stamp}_beir_scifact_retrieval.json"
output_path.write_text(_json.dumps(run_record, indent=2))

print(f"Saved: {output_path.relative_to(root)}")


Saved: benchmarks/results/20260929T113736+0200_beir_scifact_retrieval.json


## 13. Release the models


In [18]:
embedding_session.close()
